# Resumable Email Outreach Manager

A Google Colab workflow for managing individually addressed outreach emails using Google Sheets for contact tracking and Gmail API for delivery.

**Included safeguards**
- Dry-run by default; live sending is explicitly disabled.
- Persistent per-contact status tracking.
- Daily campaign cap, batch size, fixed send interval and cooldown controls.
- Suppression and opt-out fields.
- Checkpoint before and after each send.
- Interruption reconciliation for `SENDING` and `UNKNOWN` records.
- Credentials are read from Colab Secrets, never embedded in this notebook.

> **Safety:** The included sample contacts use the reserved `.test` domain and cannot receive email. All sample contacts are unapproved. Do not approve or attempt to send to sample addresses. Replace them with contacts you are authorised to contact, and follow applicable email, privacy and provider policies.


## 1. Install packages

In [ ]:
!pip -q install pandas gspread google-api-python-client google-auth google-auth-oauthlib

## 2. Imports and configuration

In [ ]:
import base64, re, time
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from email.message import EmailMessage
from html import escape
import pandas as pd, gspread, google.auth, requests
from google.colab import auth, userdata
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from IPython.display import display, HTML

IST = ZoneInfo("Asia/Kolkata")

# Configure these values for your own Google Sheet and verified sender.
SHEET_URL = "PASTE_YOUR_GOOGLE_SHEET_URL_HERE"
SOURCE_WORKSHEET = "Sheet1"
TRACKING_WORKSHEET = "Outreach_Tracking"
DAILY_LOG_WORKSHEET = "Daily_Sending_Log"

SENDER_EMAIL = "YOUR_VERIFIED_SENDER@YOUR_DOMAIN"
CC_EMAILS = []  # Optional: add authorised CC addresses.

# Conservative example defaults. Adjust only after reviewing provider limits.
DAILY_CAMPAIGN_CAP = 50
BATCH_SIZE = 25
BATCH_COOLDOWN_MINUTES = 20
FIXED_INTERVAL_SECONDS = 60
SENDING_START = "08:00"
SENDING_END = "19:00"
LIVE_SEND_ENABLED = False

TRACKING_HEADERS = [
    "Contact_ID","Source_Row","Company","Full_Name","Email","Approval_Status",
    "Send_Status","Sent_At_IST","Gmail_Message_ID","Attempt_Count",
    "Reply_Status","Opt_Out","Suppressed","Suppression_Reason","Last_Error"
]
DAILY_HEADERS = [
    "Date_IST","Sent_Count","Reserved_Count","Failed_Count",
    "Last_Updated_IST","Next_Batch_Not_Before_IST","Sent_Since_Cooldown"
]
print("Configuration ready. Live sending:", LIVE_SEND_ENABLED)


## 3. Authenticate Sheets and open source

In [ ]:
auth.authenticate_user()
sheets_creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/spreadsheets"])
gc = gspread.authorize(sheets_creds)
spreadsheet = gc.open_by_url(SHEET_URL)
source_ws = spreadsheet.worksheet(SOURCE_WORKSHEET)
print("Connected:", spreadsheet.title)

## 4. Load and validate contacts

In [ ]:
contacts_df = pd.DataFrame(source_ws.get_all_records())
contacts_df.columns = contacts_df.columns.astype(str).str.strip()
required = ["Company Name","First Name","Last Name","Title","Email"]
missing = [x for x in required if x not in contacts_df.columns]
if missing: raise ValueError(f"Missing source columns: {missing}")
contacts = contacts_df.rename(columns={
    "Company Name":"company","First Name":"first_name","Last Name":"last_name",
    "Title":"designation","Email":"email"
}).copy()
for c in ["company","first_name","last_name","designation","email"]:
    contacts[c] = contacts[c].fillna("").astype(str).str.strip()
contacts["email"] = contacts["email"].str.lower()
contacts["full_name"] = (contacts["first_name"]+" "+contacts["last_name"]).str.strip()
contacts["source_row"] = contacts.index + 2
contacts["contact_id"] = contacts["source_row"].astype(str)+"|"+contacts["email"]
EMAIL_RE = re.compile(r"^[^\s@]+@[^\s@]+\.[^\s@]+$")
contacts["email_valid"] = contacts["email"].map(lambda x: bool(EMAIL_RE.match(x)))
print("Contacts:",len(contacts),"Valid syntax:",int(contacts.email_valid.sum()))
print("Invalid/blank:",int((~contacts.email_valid).sum()))
display(contacts.loc[~contacts.email_valid,["source_row","company","full_name","email"]].head(20))

## 5. Define the HTML and plain-text email templates

In [ ]:
EMAIL_SUBJECT = "Invitation to Participate | [Your Study or Initiative]"

def build_email_html(c):
    name = escape(c.get("first_name","").strip() or "there")
    company = escape(c.get("company","").strip() or "your organisation")
    return f"""<!doctype html><html><body style="font-family:Arial,sans-serif;color:#303846;font-size:14px;line-height:1.55">
<div style="max-width:660px;margin:auto;padding:22px">
<p>Hi {name},</p>
<p>I'm reaching out from <b>[Your Organisation]</b>.</p>
<p>We would like to invite <b>{company}</b> to participate in <b>[Your Study or Initiative]</b>.</p>
<p>[Briefly explain the purpose, audience and value of the initiative. Keep claims accurate and relevant to the recipient.]</p>
<div style="background:#f3f6fa;border-left:4px solid #244b75;padding:14px 18px;margin:18px 0">
<p><b>What participation involves</b></p>
<ul><li>[Participation step one]</li><li>[Participation step two]</li><li>[Deadline, if applicable]</li></ul>
</div>
<p><a href="https://example.org/" style="display:inline-block;background:#24558a;color:white;text-decoration:none;padding:12px 22px;border-radius:4px;font-weight:bold">Learn more</a></p>
<p>Best regards,<br><b>[Your Name]</b><br>[Your Role] | [Your Organisation]</p>
<p style="font-size:11px;color:#666">If this invitation is not relevant, reply to this email and we will update our contact records.</p>
</div></body></html>"""

def build_email_text(c):
    name = c.get("first_name","").strip() or "there"
    company = c.get("company","").strip() or "your organisation"
    return f"""Hi {name},

I'm reaching out from [Your Organisation].

We would like to invite {company} to participate in [Your Study or Initiative].

[Briefly explain the purpose, audience and value.]

Participation involves:
1. [Step one]
2. [Step two]
3. [Deadline, if applicable]

More information: https://example.org/

Best regards,
[Your Name]
[Your Role] | [Your Organisation]

If this invitation is not relevant, reply to this email and we will update our contact records.
"""
print("Generic HTML and plain-text email templates loaded.")


## 6. Preview the email

In [ ]:
sample = contacts.loc[contacts.email_valid].iloc[0].to_dict()
display(HTML(build_email_html(sample)))
print("CC on vendor emails:",CC_EMAILS)

## 7. Open tracking and daily log safely

In [ ]:
def open_or_create_worksheet(title, rows=2000, cols=20):
    try:
        return spreadsheet.worksheet(title)
    except gspread.WorksheetNotFound:
        return spreadsheet.add_worksheet(title=title, rows=rows, cols=cols)

tracking_ws = open_or_create_worksheet(TRACKING_WORKSHEET)
daily_log_ws = open_or_create_worksheet(DAILY_LOG_WORKSHEET)

existing = tracking_ws.get_all_values()
if not existing or not any(str(x).strip() for r in existing for x in r):
    tracking_ws.update(range_name="A1", values=[TRACKING_HEADERS])
    init_rows = [
        [c.contact_id, int(c.source_row), c.company, c.full_name, c.email,
         "NOT_APPROVED", "NOT_SENT", "", "", 0,
         "NO_REPLY_RECORDED", "FALSE", "FALSE", "", ""]
        for c in contacts.itertuples()
    ]
    if init_rows:
        tracking_ws.append_rows(init_rows, value_input_option="RAW")
else:
    missing = [h for h in TRACKING_HEADERS if h not in existing[0]]
    if missing:
        raise RuntimeError(f"Existing tracker missing headers {missing}; no data overwritten.")

dvals = daily_log_ws.get_all_values()
if not dvals or not any(str(x).strip() for r in dvals for x in r):
    if daily_log_ws.col_count < len(DAILY_HEADERS):
        daily_log_ws.add_cols(len(DAILY_HEADERS) - daily_log_ws.col_count)
    daily_log_ws.update(range_name="A1", values=[DAILY_HEADERS])
else:
    if dvals[0][:6] != DAILY_HEADERS[:6]:
        raise RuntimeError(
            f"Daily log schema mismatch. Found {dvals[0][:6]}; "
            "expected the original six headers. No data overwritten."
        )
    if daily_log_ws.col_count < len(DAILY_HEADERS):
        daily_log_ws.add_cols(len(DAILY_HEADERS) - daily_log_ws.col_count)
    seventh = dvals[0][6].strip() if len(dvals[0]) > 6 else ""
    if not seventh:
        daily_log_ws.update_cell(1, 7, DAILY_HEADERS[6])
    elif seventh != DAILY_HEADERS[6]:
        raise RuntimeError(f"Unexpected daily log column G header: {seventh}")

print("Tracking rows:", max(0, len(tracking_ws.get_all_values()) - 1))
print("Daily log connected. Existing records preserved.")


## 8. Gmail OAuth using the previous Colab Secrets

In [ ]:
client_id = userdata.get("GMAIL_CLIENT_ID")
client_secret = userdata.get("GMAIL_CLIENT_SECRET")
refresh_token = userdata.get("GMAIL_REFRESH_TOKEN")
if not all([client_id,client_secret,refresh_token]):
    raise ValueError("Missing one of GMAIL_CLIENT_ID / GMAIL_CLIENT_SECRET / GMAIL_REFRESH_TOKEN.")
gmail_creds = Credentials(token=None,refresh_token=refresh_token,
    token_uri="https://oauth2.googleapis.com/token",
    client_id=client_id,client_secret=client_secret)
gmail_creds.refresh(Request())
gmail_service = build("gmail","v1",credentials=gmail_creds)
print("Gmail OAuth refreshed.")

## 9. Verify Gmail send scope

In [ ]:
token_info = requests.get("https://oauth2.googleapis.com/tokeninfo",
    params={"access_token":gmail_creds.token},timeout=20).json()
scopes = token_info.get("scope","").split()
if "https://www.googleapis.com/auth/gmail.send" not in scopes:
    raise PermissionError(f"gmail.send scope missing. Granted: {scopes}")
print("gmail.send scope verified.")

## 10. Gmail MIME builder and test-to-self

In [ ]:
def make_raw(to_email,subject,text_body,html_body,cc=None):
    msg = EmailMessage()
    msg["To"] = to_email
    msg["From"] = SENDER_EMAIL
    msg["Subject"] = subject
    if cc:
        msg["Cc"] = ", ".join(cc)
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")
    return base64.urlsafe_b64encode(msg.as_bytes()).decode("utf-8")

# Optional test-to-self. Disabled by default so "Run all" sends no email.
SEND_TEST_EMAIL = False

if SEND_TEST_EMAIL:
    TEST_CONTACT = sample
    test_raw = make_raw(
        SENDER_EMAIL,
        "[TEST] " + EMAIL_SUBJECT,
        build_email_text(TEST_CONTACT),
        build_email_html(TEST_CONTACT)
    )
    test_result = gmail_service.users().messages().send(
        userId="me",
        body={"raw": test_raw}
    ).execute()
    print("Test email sent to your configured sender inbox. Message ID:", test_result.get("id"))
else:
    print("Test email disabled. No email sent by this cell.")


## 11. Tracking data helpers

In [ ]:
def load_tracker():
    d = pd.DataFrame(tracking_ws.get_all_records())
    if d.empty: raise RuntimeError("Tracking sheet is empty.")
    d["Contact_ID"] = d["Contact_ID"].astype(str)
    for col in ["Approval_Status","Send_Status","Opt_Out","Suppressed"]:
        d[col] = d[col].fillna("").astype(str).str.strip().str.upper()
    return d

def update_tracker(contact_id,updates):
    vals = tracking_ws.get_all_values()
    headers = vals[0]
    hmap = {h:i+1 for i,h in enumerate(headers)}
    rowmap = {str(r[0]):i+2 for i,r in enumerate(vals[1:]) if r and r[0]}
    if contact_id not in rowmap: raise KeyError(f"Unknown Contact_ID {contact_id}")
    for k,v in updates.items():
        if k not in TRACKING_HEADERS: raise ValueError(k)
        tracking_ws.update_cell(rowmap[contact_id],hmap[k],v)

def sync_queue():
    tr = load_tracker()
    lookup = tr.set_index("Contact_ID")
    q = contacts.copy()
    for sheet_col,local_col,default in [
        ("Approval_Status","approval_status","NOT_APPROVED"),
        ("Send_Status","send_status","NOT_SENT"),
        ("Opt_Out","opt_out","FALSE"),
        ("Suppressed","suppressed","FALSE"),
    ]:
        q[local_col] = q.contact_id.map(lookup[sheet_col]).fillna(default)
    return q,tr

outreach_queue,tracker_df = sync_queue()
print("Approved:",int((outreach_queue.approval_status=="APPROVED").sum()))
print("Already sent:",int((outreach_queue.send_status=="SENT").sum()))

## 12. Daily ledger and campaign cap

In [ ]:
def now_ist(): return datetime.now(IST)

def today_row():
    today = now_ist().strftime("%Y-%m-%d")
    vals = daily_log_ws.get_all_values()
    for n,row in enumerate(vals[1:],start=2):
        if row and row[0]==today: return n
    daily_log_ws.append_row([today,0,0,0,now_ist().strftime("%Y-%m-%d %H:%M:%S"),"",0],value_input_option="RAW")
    return len(daily_log_ws.get_all_values())

def daily_usage():
    n=today_row()
    r=daily_log_ws.row_values(n)+[""]*7
    sent,reserved,failed=int(r[1] or 0),int(r[2] or 0),int(r[3] or 0)
    return {"sent":sent,"reserved":reserved,"failed":failed,
            "remaining":max(0,DAILY_CAMPAIGN_CAP-sent-reserved),
            "cooldown_until":r[5],"sent_since_cooldown":int(r[6] or 0)}

def ledger_update(sent=0,reserved=0,failed=0,cooldown=None,since_break=None):
    n=today_row(); u=daily_usage(); r=daily_log_ws.row_values(n)+[""]*7
    new_since = u["sent_since_cooldown"] if since_break is None else since_break
    daily_log_ws.update(range_name=f"A{n}:G{n}",values=[[
        now_ist().strftime("%Y-%m-%d"),max(0,u["sent"]+sent),
        max(0,u["reserved"]+reserved),max(0,u["failed"]+failed),
        now_ist().strftime("%Y-%m-%d %H:%M:%S"),
        cooldown if cooldown is not None else r[5],new_since
    ]],value_input_option="RAW")

print("Daily usage:",daily_usage())

## 13. Eligibility, suppression and uncertainty checks

In [ ]:
def get_eligible():
    q,tr=sync_queue()
    uncertain=tr[tr.Send_Status.isin(["SENDING","UNKNOWN"])]
    if not uncertain.empty:
        display(uncertain[["Contact_ID","Company","Email","Send_Status","Last_Error"]])
        raise RuntimeError("Paused: SENDING/UNKNOWN records require manual Gmail Sent reconciliation.")
    mask=(q.email_valid &
          q.approval_status.eq("APPROVED") &
          q.send_status.eq("NOT_SENT") &
          q.opt_out.isin(["FALSE","","0"]) &
          q.suppressed.isin(["FALSE","","0"]) &
          ~q.email.duplicated(keep="first"))
    return q[mask].copy()

print("Eligible:",len(get_eligible()))

## 14. Preview next batch

In [ ]:
def preview_batch():
    eligible=get_eligible(); u=daily_usage()
    room_to_break=max(0,BATCH_SIZE-u["sent_since_cooldown"])
    batch=eligible.head(min(BATCH_SIZE,room_to_break,u["remaining"]))
    print("Eligible:",len(eligible),"Selected:",len(batch))
    print("Remaining until next batch cooldown:",room_to_break)
    print("Daily ledger:",u)
    display(batch[["source_row","company","full_name","email"]])
    return batch

preview_batch()

## 15. Window and cooldown checks

In [ ]:
def in_window():
    t=now_ist().strftime("%H:%M")
    return SENDING_START <= t <= SENDING_END

def cooldown_complete():
    value=daily_usage()["cooldown_until"]
    return not value or now_ist() >= datetime.fromisoformat(value)

print("Within sending window:",in_window())
print("Cooldown complete:",cooldown_complete())

## 16. Gmail sender

In [ ]:
def send_contact(c):
    raw=make_raw(c["email"],EMAIL_SUBJECT,build_email_text(c),build_email_html(c),cc=CC_EMAILS)
    return gmail_service.users().messages().send(userId="me",body={"raw":raw}).execute()

## 17. Batch execution — checkpoint before and after every send

In [ ]:
def run_batch(dry_run=True):
    if not dry_run and not LIVE_SEND_ENABLED:
        raise RuntimeError("Live sending is disabled. Explicitly enable only after review.")
    if not in_window():
        print("Outside sending window; no send.")
        return 0
    if not cooldown_complete():
        print("Cooldown active until:",daily_usage()["cooldown_until"])
        return 0

    batch=preview_batch()
    if batch.empty: return 0
    if dry_run:
        print("DRY RUN ONLY — no messages sent or records changed.")
        return 0

    usage_now=daily_usage()
    room_to_break=BATCH_SIZE-usage_now["sent_since_cooldown"]
    count=min(len(batch),room_to_break)
    batch=batch.head(count)
    print("Selected until next cooldown:",count)
    ledger_update(reserved=count)
    sent_now=0
    try:
        for pos,(_,c) in enumerate(batch.iterrows(),start=1):
            if not in_window():
                print("Sending window ended.")
                break
            tr=load_tracker().set_index("Contact_ID")
            old_attempt=int(tr.loc[c.contact_id,"Attempt_Count"] or 0)
            update_tracker(c.contact_id,{"Send_Status":"SENDING","Attempt_Count":old_attempt+1,"Last_Error":""})
            try:
                result=send_contact(c.to_dict())
            except Exception as e:
                update_tracker(c.contact_id,{"Send_Status":"UNKNOWN","Last_Error":f"{type(e).__name__}: {str(e)[:700]}"})
                ledger_update(reserved=-count,failed=1)
                raise RuntimeError("Send outcome uncertain. Batch stopped. Reconcile this contact in Gmail Sent before retrying.") from e

            update_tracker(c.contact_id,{
                "Send_Status":"SENT",
                "Sent_At_IST":now_ist().strftime("%Y-%m-%d %H:%M:%S"),
                "Gmail_Message_ID":result.get("id",""),
                "Last_Error":""
            })
            current_since=daily_usage()["sent_since_cooldown"]+1
            sent_now+=1
            if current_since >= BATCH_SIZE:
                until=(now_ist()+timedelta(minutes=BATCH_COOLDOWN_MINUTES)).isoformat()
                ledger_update(sent=1,reserved=-1,cooldown=until,since_break=0)
                print(f"{BATCH_SIZE}-message threshold reached. Cooldown until:", until)
            else:
                ledger_update(sent=1,reserved=-1,since_break=current_since)
            print(f"SENT {sent_now}/{count}: {c.email}")
            if pos<count and current_since < BATCH_SIZE: time.sleep(FIXED_INTERVAL_SECONDS)
        # Release only the unused reservation for contacts never attempted.
        remaining=daily_usage()["reserved"]
        if remaining: ledger_update(reserved=-remaining)
        return sent_now
    except KeyboardInterrupt:
        # If interruption occurred during a Gmail request, its SENDING checkpoint
        # remains and will block future sends until manually reconciled.
        # Release only the unused campaign reservation; never change send statuses.
        remaining=daily_usage()["reserved"]
        if remaining: ledger_update(reserved=-remaining)
        print("Interrupted. Unused reservation released. Reconcile SENDING/UNKNOWN before resuming.")
        raise
    except Exception:
        # Do not auto-retry. Leave SENT/SENDING/UNKNOWN checkpoints intact.
        remaining=daily_usage()["reserved"]
        if remaining: ledger_update(reserved=-remaining)
        raise

print("Execution engine loaded. No emails sent by defining this function.")

## 18. Reconcile after interruption — read-only

In [ ]:
def reconcile():
    tr=load_tracker()
    print(tr.Send_Status.value_counts(dropna=False))
    uncertain=tr[tr.Send_Status.isin(["SENDING","UNKNOWN"])]
    if uncertain.empty: print("No uncertain records.")
    else:
        print("STOP: manually search Gmail Sent by recipient/time before deciding status.")
        display(uncertain[["Contact_ID","Company","Email","Send_Status","Last_Error"]])
    print("Daily ledger:",daily_usage())
    print("Read-only; no email sent and no tracker modified.")

reconcile()

## 19. Resume preview and send functions

In [ ]:
def resume_preview():
    return preview_batch()

def resume_send():
    return run_batch(dry_run=False)

print("Resume helpers ready. Always call resume_preview() first.")

## 20. Explicit live-send switch — keep OFF until approved

In [ ]:
# Keep False for dry runs. Set True only after you have reviewed:
# - the recipient preview,
# - the test email,
# - the source data and approval statuses,
# - opt-out/suppression handling,
# - applicable Gmail and legal sending requirements.
LIVE_SEND_ENABLED = False
print("LIVE_SEND_ENABLED =", LIVE_SEND_ENABLED)
print("This cell only changes the safety switch; it does not send email.")


## 21. Explicit live execution (optional)\n\nThis cell is intentionally separate from the live-send switch. It will send only if `LIVE_SEND_ENABLED` is explicitly set to `True`, the contacts are approved, the cooldown and daily cap permit sending, and the sending window is open. Keep this cell unrun for a dry-run demo.

In [ ]:
# LIVE ACTION: this calls the Gmail API when enabled.
if not LIVE_SEND_ENABLED:
    raise RuntimeError("Live sending is disabled. Keep this off for the sample workbook.")

print("Starting live outreach batch...")
sent_count = resume_send()
print(f"Batch finished. Gmail accepted {sent_count} message(s).")


## 22. Campaign status

In [ ]:
def campaign_status():
    tr=load_tracker()
    print("Total:",len(tr))
    print("Approved:",int((tr.Approval_Status=="APPROVED").sum()))
    print("Sent:",int((tr.Send_Status=="SENT").sum()))
    print("Not sent:",int((tr.Send_Status=="NOT_SENT").sum()))
    print("Uncertain:",int(tr.Send_Status.isin(["SENDING","UNKNOWN"]).sum()))
    print("Opt-outs:",int((tr.Opt_Out=="TRUE").sum()))
    print("Suppressed:",int((tr.Suppressed=="TRUE").sum()))
    print("Daily ledger:",daily_usage())
campaign_status()

## Resume procedure after interruption

**Same connected Colab runtime:** run reconciliation, inspect any `SENDING` or `UNKNOWN` contacts, confirm cooldown and preview, then call `resume_send()` again.

**Fresh Colab runtime:** rerun setup cells in order to restore packages, Google Sheets authentication, source data, tracking, Gmail OAuth and function definitions. Then reconcile, preview, confirm the live switch is still explicitly enabled only if intended, and execute the live-send cell.

The Google Sheet is the persistent source of truth for contact statuses. A successful Gmail API response is recorded as `SENT`; it does not guarantee inbox placement. If a send outcome is uncertain, do not retry blindly.

**Colab is resumable, not unattended.** A disconnected runtime stops execution. For scheduled unattended workflows, use a properly secured cloud service with appropriate monitoring and controls.

**Sending limits:** The configured daily cap is an application-level cap, not a promise that Gmail permits that volume. Google limits depend on account type and policy. Start conservatively, comply with applicable laws and provider terms, and monitor bounces, complaints and errors.
